In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    Multiply, concatenate, Dot
)
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Environment Setup ─────────────────────────────────────────────────────────
# Kaggle  : set USE_AUGMENTED = True/False sesuai kebutuhan
# Lokal   : ubah DATA_DIR ke path lokal

DATASET_SLUG  = "siamese-data"   # <-- GANTI sesuai nama dataset Kaggle
DATA_DIR      = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR       = "/kaggle/working"
USE_AUGMENTED = True             # True  → pakai aug_*.npy + aug_metadata.pkl
                                 # False → pakai data asli

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

prefix = 'final_' if USE_AUGMENTED else ''
meta_f = 'final_metadata.pkl' if USE_AUGMENTED else 'metadata.pkl'
for fname in [f'{prefix}questions_emb.npy', f'{prefix}answerkeys_emb.npy',
              f'{prefix}answers_emb.npy',    meta_f]:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<30} -> {status}")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
# final_questions_emb & final_answerkeys_emb disimpan KOMPAK (1 per IDPSJ).
# Rekonstruksi array penuh menggunakan kolom psj_idx di metadata.

if USE_AUGMENTED:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
else:
    answers_emb    = np.load(os.path.join(DATA_DIR, 'answers_emb.npy'))
    uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'questions_emb.npy'))
    uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'answerkeys_emb.npy'))
    metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'metadata.pkl'))

metadata = metadata.reset_index(drop=True)

# Jika data asli (belum kompak), psj_idx belum ada → buat sekarang
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    idpsj_to_idx = {psj: i for i, psj in enumerate(idpsj_sorted)}
    metadata['psj_idx'] = metadata['IDPSJ'].map(idpsj_to_idx)

# Rekonstruksi array penuh menggunakan psj_idx
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print("=== Hasil Load ===")
print(f"answers_emb    : {answers_emb.shape}    (per sampel)")
print(f"uniq_q_emb     : {uniq_q_emb.shape}   (kompak, per IDPSJ)")
print(f"questions_emb  : {questions_emb.shape}  (rekonstruksi)")
print(f"answerkeys_emb : {answerkeys_emb.shape} (rekonstruksi)")
print(f"\nMetadata       : {len(metadata)} rows")
print(f"Kolom metadata : {list(metadata.columns)}")
print(f"\nIDPSJ unik     : {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

In [ ]:
# ── Hyperparameter (tuned via Optuna, best MAE=1.8615 on 4-fold proxy) ────────
BILSTM_UNITS  = 128
DROPOUT       = 0.40
EPOCHS        = 150
BATCH_SIZE    = 16
PATIENCE      = 15
LR            = 2.68e-3

# ── Ordinal Regression Loss & Metric ─────────────────────────────────────────
def ordinal_loss(y_true, y_pred):
    """Sum of binary cross-entropies: apakah grade > k? (k=1..9)"""
    thresholds   = tf.cast(tf.range(1, 10), tf.float32)
    y_true_exp   = tf.expand_dims(tf.cast(y_true, tf.float32), -1)
    y_binary     = tf.cast(y_true_exp > thresholds, tf.float32)
    return tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_binary, y_pred)
    )

def ordinal_mae(y_true, y_pred):
    """Grade prediction = jumlah threshold yang terlampaui + 1."""
    grade_pred = tf.reduce_sum(tf.cast(y_pred > 0.5, tf.float32), axis=-1) + 1.0
    return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.float32) - grade_pred))


# ── Custom layers (pengganti Lambda agar serializable di Keras 3) ─────────────
class SoftmaxAttn(tf.keras.layers.Layer):
    def call(self, x):
        return tf.nn.softmax(x, axis=1)
    def compute_output_shape(self, s):
        return s

class AttnPool(tf.keras.layers.Layer):
    def call(self, inputs):
        seq_out, weights = inputs
        return tf.reduce_sum(seq_out * weights, axis=1)
    def compute_output_shape(self, shapes):
        return (shapes[0][0], shapes[0][2])

class AbsDiff(tf.keras.layers.Layer):
    def call(self, inputs):
        return tf.abs(inputs[0] - inputs[1])
    def compute_output_shape(self, shapes):
        return shapes[0]

class OneMinus(tf.keras.layers.Layer):
    def call(self, x):
        return 1.0 - x
    def compute_output_shape(self, s):
        return s


def attention_pool(seq_out, name_prefix):
    """Soft attention pooling atas output BiLSTM (return_sequences=True)."""
    score   = Dense(1, activation='tanh', use_bias=False,
                    name=f'{name_prefix}_attn_score')(seq_out)
    weights = SoftmaxAttn(name=f'{name_prefix}_attn_w')(score)
    pooled  = AttnPool(name=f'{name_prefix}_attn_pool')([seq_out, weights])
    return pooled


def build_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300,
                n_scalar=4,
                bilstm_units=BILSTM_UNITS, dropout=DROPOUT):
    """
    Siamese BiLSTM v11 — Ordinal Regression
    4 scalar features (f1_pre, pre, f1, cos) — tanpa lrat dan rec.
    lrat dihapus: penalti panjang eksplisit mengalahkan sinyal BiLSTM.
    rec  dihapus: coverage-based, menghukum jawaban pendek tapi benar.

    Merged: [ea(256), eak(256), eq(256), abs_diff(256), had_prod(256),
             cos_sim_ak_a(1), cos_sim_q_a(1), orisinalitas(1),
             scalar_dense(32)]  = 1315D
    Head: Dense(512) → Dropout → Dense(64) → Dense(9, sigmoid)  [ordinal]
    """
    shared_bilstm = Bidirectional(
        LSTM(bilstm_units, return_sequences=True), name='bilstm_shared'
    )

    inp_q      = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak     = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a      = Input(shape=(a_seq_len,  emb_dim), name='inp_a')
    inp_scalar = Input(shape=(n_scalar,),            name='inp_scalar')

    eq_seq  = shared_bilstm(inp_q)
    eak_seq = shared_bilstm(inp_ak)
    ea_seq  = shared_bilstm(inp_a)

    eq  = attention_pool(eq_seq,  'q')
    eak = attention_pool(eak_seq, 'ak')
    ea  = attention_pool(ea_seq,  'a')

    abs_diff     = AbsDiff(name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])
    cos_sim_q_a  = Dot(axes=1, normalize=True, name='cos_sim_q_a')([eq, ea])
    orisinalitas = OneMinus(name='orisinalitas')(cos_sim_q_a)

    scalar_feat = Dense(32, activation='relu', name='scalar_dense')(inp_scalar)

    merged = concatenate(
        [ea, eak, eq, abs_diff, had_prod,
         cos_sim_ak_a, cos_sim_q_a, orisinalitas,
         scalar_feat],
        name='merged'
    )

    x   = Dense(512, activation='relu')(merged)
    x   = Dropout(dropout)(x)
    x   = Dense(64, activation='relu')(x)
    out = Dense(9, activation='sigmoid', name='ordinal_out')(x)

    model = Model(
        inputs=[inp_q, inp_ak, inp_a, inp_scalar],
        outputs=out,
        name='siamese_bilstm_v11'
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss=ordinal_loss,
        metrics=[ordinal_mae]
    )
    return model


_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2],
    n_scalar   = 4
)
_tmp.summary()
del _tmp

In [ ]:
# ── Helper functions + Precompute Scalar Features ─────────────────────────────
from sklearn.metrics import cohen_kappa_score

def compute_scalar_features_all(answers_emb, answerkeys_emb, verbose=True):
    """
    4 scalar features — tanpa lrat dan rec:
      f1_pre : precision-biased F1 (β=0.5) — rewards jawaban on-topic tanpa
               menghukum jawaban pendek yang tidak mencakup semua token kunci
      pre    : soft-ROUGE precision (setiap token jawaban relevan ke kunci?)
      f1     : balanced soft-ROUGE F1
      cos    : cosine similarity antar centroid

    Dihapus vs versi lama:
      rec  → mendominasi penalti pada jawaban pendek (coverage answerkey)
      lrat → penalti panjang eksplisit, mengalahkan sinyal BiLSTM
    """
    n = answers_emb.shape[0]
    feats = np.zeros((n, 4), dtype=np.float32)
    if verbose:
        print(f"Precomputing scalar features untuk {n} sampel...")
    for i in range(n):
        if verbose and i % 500 == 0:
            print(f"  {i}/{n}")
        a  = answers_emb[i].astype(np.float64)
        ak = answerkeys_emb[i].astype(np.float64)

        ak_norm = ak / (np.linalg.norm(ak, axis=-1, keepdims=True) + 1e-8)
        a_norm  = a  / (np.linalg.norm(a,  axis=-1, keepdims=True) + 1e-8)

        ak_mask = np.abs(ak).sum(axis=-1) > 1e-6
        a_mask  = np.abs(a ).sum(axis=-1) > 1e-6
        ak_n    = ak_norm[ak_mask]
        a_n     = a_norm[a_mask]

        if ak_n.shape[0] == 0 or a_n.shape[0] == 0:
            continue

        sim = ak_n @ a_n.T                        # (T_ak, T_a)
        rec = float(sim.max(axis=1).mean())        # dipakai untuk hitung f1 saja
        pre = float(sim.max(axis=0).mean())
        f1  = 2.0 * rec * pre / (rec + pre + 1e-8)

        # Precision-biased F1 (β=0.5): precision dihitung 4× lebih penting dari recall.
        # Untuk jawaban pendek tapi benar: pre tinggi, rec rendah → f1_pre lebih tinggi dari f1.
        beta    = 0.5
        f1_pre  = (1 + beta**2) * pre * rec / (beta**2 * pre + rec + 1e-8)

        m_ak = ak_n.mean(axis=0)
        m_a  = a_n.mean(axis=0)
        cos  = float(m_ak @ m_a / (np.linalg.norm(m_ak) * np.linalg.norm(m_a) + 1e-8))

        feats[i] = [f1_pre, pre, f1, cos]

    if verbose:
        print(f"  Selesai. Shape: {feats.shape}")
    return feats


def ordinal_predict(model, X_q, X_ak, X_a, X_scalar=None):
    inputs = [X_q, X_ak, X_a]
    if X_scalar is not None:
        inputs.append(X_scalar)
    sigmoid_out = model.predict(inputs, verbose=0)
    grade = np.sum(sigmoid_out > 0.5, axis=-1) + 1
    return grade.astype(np.float32), sigmoid_out


# ── Precompute scalar features (1x untuk semua sampel) ───────────────────────
all_scalar_feats = compute_scalar_features_all(answers_emb, answerkeys_emb)
# 4 fitur: f1_pre, pre, f1, cos  (tanpa lrat dan rec)
N_SCALAR = all_scalar_feats.shape[1]   # 4

# ── Fixed Split Setup ─────────────────────────────────────────────────────────
TEST_IDPSJ  = [6, 14]
VAL_IDPSJ   = 16

idpsj_list  = sorted(metadata['IDPSJ'].unique())
y_all       = metadata['grade'].values.astype(np.float32)

if 'is_synthetic' in metadata.columns:
    is_real = ~metadata['is_synthetic'].values
else:
    is_real = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_')).values

TRAIN_IDPSJ = [pid for pid in idpsj_list if pid not in TEST_IDPSJ and pid != VAL_IDPSJ]

train_idx = metadata.index[metadata['IDPSJ'].isin(TRAIN_IDPSJ)].values
val_idx   = metadata.index[(metadata['IDPSJ'] == VAL_IDPSJ) & is_real].values
test_idx  = metadata.index[metadata['IDPSJ'].isin(TEST_IDPSJ) & is_real].values

print(f"N_SCALAR={N_SCALAR} | {len(idpsj_list)} IDPSJ | {is_real.sum()} sampel real")
print(f"Train : {len(TRAIN_IDPSJ)} IDPSJ = {TRAIN_IDPSJ}")
print(f"Val   : IDPSJ {VAL_IDPSJ} | {len(val_idx)} sampel real")
print(f"Test  : IDPSJ {TEST_IDPSJ} | {len(test_idx)} sampel real")

In [ ]:
# ── Training Fixed Split ───────────────────────────────────────────────────────
y_train = y_all[train_idx]
y_val   = y_all[val_idx]
y_test  = y_all[test_idx]

print(f"Train: {len(y_train)} sampel | Val: {len(y_val)} sampel | Test: {len(y_test)} sampel")

def get_split(arr, idx):
    return arr[idx].astype(np.float32)

X_q_tr   = get_split(questions_emb,  train_idx)
X_ak_tr  = get_split(answerkeys_emb, train_idx)
X_a_tr   = get_split(answers_emb,    train_idx)
X_q_val  = get_split(questions_emb,  val_idx)
X_ak_val = get_split(answerkeys_emb, val_idx)
X_a_val  = get_split(answers_emb,    val_idx)
X_q_te   = get_split(questions_emb,  test_idx)
X_ak_te  = get_split(answerkeys_emb, test_idx)
X_a_te   = get_split(answers_emb,    test_idx)

# Gunakan 5 fitur raw tanpa normalisasi — konsisten dengan inference di LMS
scalar_tr  = all_scalar_feats[train_idx].astype(np.float32)
scalar_val = all_scalar_feats[val_idx].astype(np.float32)
scalar_te  = all_scalar_feats[test_idx].astype(np.float32)

# Sample weights (inverse class frequency)
grade_int          = y_train.astype(int)
unique_g, counts_g = np.unique(grade_int, return_counts=True)
freq_map           = dict(zip(unique_g, counts_g))
n_kelas            = len(unique_g)
raw_w              = np.array([len(y_train) / (n_kelas * freq_map[g]) for g in grade_int])
sample_w           = raw_w / raw_w.mean()
print(f"Sample weight  min={sample_w.min():.2f}  max={sample_w.max():.2f}  mean={sample_w.mean():.2f}")

# ── Training ──────────────────────────────────────────────────────────────────
tf.random.set_seed(42)
np.random.seed(42)

model = build_model(
    q_seq_len    = questions_emb.shape[1],
    ak_seq_len   = answerkeys_emb.shape[1],
    a_seq_len    = answers_emb.shape[1],
    emb_dim      = answers_emb.shape[2],
    n_scalar     = N_SCALAR,
    bilstm_units = BILSTM_UNITS,
    dropout      = DROPOUT
)

model_path = os.path.join(OUT_DIR, 'model_fixed_split.keras')
callbacks = [
    EarlyStopping(monitor='val_ordinal_mae', patience=PATIENCE,
                  restore_best_weights=True, mode='min', verbose=1),
    ReduceLROnPlateau(monitor='val_ordinal_mae', factor=0.5,
                      patience=3, min_lr=1e-6, mode='min', verbose=0),
]

model.fit(
    [X_q_tr, X_ak_tr, X_a_tr, scalar_tr], y_train,
    sample_weight=sample_w,
    validation_data=([X_q_val, X_ak_val, X_a_val, scalar_val], y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=callbacks, verbose=1
)

# ── Prediksi & Evaluasi ───────────────────────────────────────────────────────
y_pred_final, _ = ordinal_predict(model, X_q_te, X_ak_te, X_a_te, scalar_te)
y_pred_final    = np.clip(y_pred_final, 1, 10)

mae_final  = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
qwk_final  = cohen_kappa_score(y_test.astype(int), y_pred_final.astype(int),
                                weights='quadratic')

print(f"\n{'='*60}")
print(f"Hasil Final")
print(f"Test IDPSJ : {TEST_IDPSJ}")
print(f"MAE  : {mae_final:.4f}")
print(f"RMSE : {rmse_final:.4f}")
print(f"QWK  : {qwk_final:.4f}")

model.save(model_path)
print(f"\nModel disimpan -> {model_path}")

In [ ]:
# ── Evaluasi Akhir ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

print("=" * 60)
print("Hasil — Siamese BiLSTM v11 (Fixed Split)")
print("=" * 60)
print(f"Train : {len(TRAIN_IDPSJ)} IDPSJ = {TRAIN_IDPSJ}")
print(f"Val   : IDPSJ {VAL_IDPSJ}")
print(f"Test  : IDPSJ {TEST_IDPSJ}")
print(f"\nMAE   : {mae_final:.4f}")
print(f"RMSE  : {rmse_final:.4f}")
print(f"QWK   : {qwk_final:.4f}")

# ── Per-IDPSJ breakdown di test ───────────────────────────────────────────────
test_meta_view = metadata.iloc[test_idx].copy().reset_index(drop=True)
test_meta_view['grade_pred'] = y_pred_final.astype(int)
test_meta_view['error']      = np.abs(y_test - y_pred_final).astype(int)

print("\n── Per-IDPSJ Test Breakdown ──")
per_idpsj_rows = []
for idpsj in sorted(TEST_IDPSJ):
    mask   = test_meta_view['IDPSJ'] == idpsj
    y_t    = y_test[mask.values]
    y_p    = y_pred_final[mask.values]
    mae_i  = mean_absolute_error(y_t, y_p)
    rmse_i = np.sqrt(mean_squared_error(y_t, y_p))
    qwk_i  = cohen_kappa_score(y_t.astype(int), y_p.astype(int), weights='quadratic')
    per_idpsj_rows.append({'IDPSJ': idpsj, 'n': mask.sum(), 'MAE': mae_i,
                            'RMSE': rmse_i, 'QWK': qwk_i})
    print(f"  IDPSJ {idpsj:2d}: n={mask.sum():3d}  MAE={mae_i:.4f}  RMSE={rmse_i:.4f}  QWK={qwk_i:.4f}")

# ── Simpan hasil ke CSV ───────────────────────────────────────────────────────
results_df = pd.DataFrame({
    'IDPSJ'     : test_meta_view['IDPSJ'].values,
    'IDJwb'     : test_meta_view['IDJwb'].values,
    'grade'     : y_test.astype(int),
    'grade_pred': y_pred_final.astype(int),
    'error'     : np.abs(y_test - y_pred_final).astype(int),
})
csv_path = os.path.join(OUT_DIR, 'fixed_split_results.csv')
results_df.to_csv(csv_path, index=False)
print(f"\nHasil disimpan -> {csv_path}")

per_df = pd.DataFrame(per_idpsj_rows)
colors = ['steelblue', 'darkorange']
color_map = {idpsj: c for idpsj, c in zip(sorted(TEST_IDPSJ), colors)}

# ── Gambar 1: Scatter plot prediksi vs aktual ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
for idpsj in sorted(TEST_IDPSJ):
    mask = test_meta_view['IDPSJ'].values == idpsj
    ax.scatter(y_test[mask], y_pred_final[mask], alpha=0.5,
               edgecolors='k', linewidths=0.3,
               label=f'IDPSJ {idpsj}', color=color_map[idpsj])
ax.plot([1, 10], [1, 10], 'r--', label='Ideal')
ax.set_xlabel('Grade Aktual')
ax.set_ylabel('Grade Prediksi')
ax.set_title('Prediksi vs Aktual — Fixed Split')
ax.set_xticks(range(1, 11))
ax.set_yticks(range(1, 11))
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fixed_split_scatter.png'), dpi=150)
plt.show()

# ── Gambar 2: Distribusi error ────────────────────────────────────────────────
errors = np.abs(y_test - y_pred_final).astype(int)
error_counts = pd.Series(errors).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(error_counts.index, error_counts.values,
              color='steelblue', edgecolor='black', linewidth=0.5)
ax.bar_label(bars, fontsize=10, fontweight='bold', padding=3)
ax.set_xlabel('|Grade Aktual - Grade Prediksi|')
ax.set_ylabel('Jumlah Sampel')
ax.set_title(f'Distribusi Error — Fixed Split (Test IDPSJ {TEST_IDPSJ})')
ax.set_ylim(0, error_counts.values.max() * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fixed_split_error_dist.png'), dpi=150)
plt.show()

# ── Gambar 3: MAE per IDPSJ ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
bars_mae = ax.bar(per_df['IDPSJ'].astype(str), per_df['MAE'],
                  color='steelblue', edgecolor='black', linewidth=0.5)
ax.bar_label(bars_mae, fmt='%.3f', fontsize=9, padding=3)
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('MAE')
ax.set_title(f'MAE per IDPSJ — Fixed Split  (Overall = {mae_final:.4f})')
ax.set_ylim(0, per_df['MAE'].max() * 1.2)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fixed_split_mae.png'), dpi=150)
plt.show()

# ── Gambar 4: RMSE per IDPSJ ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
bars_rmse = ax.bar(per_df['IDPSJ'].astype(str), per_df['RMSE'],
                   color='salmon', edgecolor='black', linewidth=0.5)
ax.bar_label(bars_rmse, fmt='%.3f', fontsize=9, padding=3)
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('RMSE')
ax.set_title(f'RMSE per IDPSJ — Fixed Split  (Overall = {rmse_final:.4f})')
ax.set_ylim(0, per_df['RMSE'].max() * 1.2)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fixed_split_rmse.png'), dpi=150)
plt.show()

# ── Gambar 5: QWK per IDPSJ ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
bars_qwk = ax.bar(per_df['IDPSJ'].astype(str), per_df['QWK'],
                  color='mediumpurple', edgecolor='black', linewidth=0.5)
ax.bar_label(bars_qwk, fmt='%.3f', fontsize=9, padding=3)
ax.set_xlabel('Test IDPSJ')
ax.set_ylabel('QWK')
ax.set_title(f'QWK per IDPSJ — Fixed Split  (Overall = {qwk_final:.4f})')
ax.set_ylim(0, max(per_df['QWK'].max() * 1.2, 0.2))
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fixed_split_qwk.png'), dpi=150)
plt.show()

print("\nGambar disimpan:")
for name in ['fixed_split_scatter.png', 'fixed_split_error_dist.png',
             'fixed_split_mae.png', 'fixed_split_rmse.png', 'fixed_split_qwk.png']:
    print(f"  {os.path.join(OUT_DIR, name)}")